# Sentiment corpus → Hugging Face Dataset

This workshop notebook downloads the [Opinion corpus of Slovene web commentaries KKS 1.001](https://www.clarin.si/repository/xmlui/handle/11356/1115), parses the XML annotations, and creates a Hugging Face `DatasetDict` with three fields:

- `text`
- `label`
- `label_text`


## 1. Install dependencies

In [ ]:
!pip install -q datasets huggingface_hub scikit-learn pandas

## 2. Download the corpus files

This uses the command-line download URL from the corpus page. The CLARIN entry contains:

- the full corpus zip
- the balanced corpus zip
- two accompanying PDF documents

We will download only the full corpus.

In [ ]:
!mkdir -p data
!cd data && curl -L --remote-name-all 'https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1115{/klxSAcorpus_20160224_1001.zip}'
!unzip -q data/klxSAcorpus_20160224_1001.zip -d data
!ls -lh data

Your data directory should now contain these files (if you can't see them in the File tree, try refreshing):

```text
data/
└── klxSAcorpus_20160224_1001/
    ├── annotatedData.xsd
    ├── klxSAcorpus_20160224_1001.xml
    └── readme.txt
```
We will only need the xml file for this exercise.

In [ ]:
XML_PATH = "data/klxSAcorpus_20160224_1001/klxSAcorpus_20160224_1001.xml"

## 3. Inspect the downloaded file


Each example (AnnotatedItem) is supposed to look roughly like this:

In [ ]:
import xml.etree.ElementTree as ET

root = ET.parse(XML_PATH).getroot()
first_item = next(
    item
    for item in root.iter()
    if item.tag.endswith("AnnotatedItem")
)

print(ET.tostring(first_item, encoding="unicode"))

## 4. Parse `AnnotatedItem` entries from XML





For sentiment classification, we need a dataset where each comment is paired with a sentiment label. The original XML corpus contains many fields (e.g., source, category, timestamp), but only two are needed for this task:

- `Content` → the comment text (`text`)
- `Score` → the sentiment annotation (`label`)

We also create a human-readable version of the label (`label_text`):

| Score | Label | Label text |
|---------|---------|------------|
| -1 | 0 | negative |
| 0 | 1 | neutral |
| 1 | 2 | positive |

The resulting dataset will contain three columns:

- `text` — the comment text
- `label` — the numeric class used for training
- `label_text` — the corresponding sentiment category

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Map the original sentiment scores to readable labels
score_to_text = {
    -1: "negative",
     0: "neutral",
     1: "positive",
}

# Map the original sentiment scores to class IDs
score_to_label = {
    -1: 0,
     0: 1,
     1: 2,
}

# Parse the XML corpus
root = ET.parse(XML_PATH).getroot()
rows = []

# Extract one training example from each AnnotatedItem
for item in root.iter():
    if not item.tag.endswith("AnnotatedItem"):
        continue

    # Extract the comment text
    text = next(
        child.text
        for child in item
        if child.tag.endswith("Content")
    )

    # Extract the sentiment annotation
    score = int(
        next(
            child.text
            for child in item
            if child.tag.endswith("Score")
        )
    )

    rows.append({
        "text": text.strip(),
        "label": score_to_label[score],
        "label_text": score_to_text[score],
    })

# Convert to a pandas DataFrame
df = pd.DataFrame(rows)

## 5. Validate and inspect the dataset

The dataset should contain 4777 examples with 3 possible sentiment labels, 3291 negative, 898 positive and 588 neutral.

In [ ]:
print(df.shape)
df['label_text'].value_counts()

Show a random sample of 5 items.

In [ ]:
df.sample(5)

## 6. Create train / validation / test splits

This corpus download does not provide a ready-made Hugging Face split, so we create reproducible stratified splits.

Default split:

- 80% train
- 10% validation
- 10% test

The seed makes the split deterministic.

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, ClassLabel, Features, Value

SEED = 42

# Split the data into train (80%) and temporary (20%) sets
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)

# Split the temporary set into validation (10%) and test (10%) sets
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

# Define the dataset schema
features = Features({
    "text": Value("string"),
    "label": Value("int64"),
    "label_text": Value("string"),
})

# Create a Hugging Face DatasetDict with train/validation/test splits
raw_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False, features=features),
    "validation": Dataset.from_pandas(valid_df, preserve_index=False, features=features),
    "test": Dataset.from_pandas(test_df, preserve_index=False, features=features),
})

raw_ds

Inspect one example.

In [ ]:
raw_ds['train'][1]

## 7. Check label distributions

This confirms that the stratified split preserved the class balance reasonably well.

In [ ]:
for split_name, split in raw_ds.items():
    counts = pd.Series(split['label_text']).value_counts().sort_index()
    print(f'{split_name}')
    print(counts)

## 8. Optional: push to Hugging Face Hub

Uncomment and run these cells if you want the dataset accessible through `load_dataset(...)`.

Before pushing, create or log into your Hugging Face account and make sure you have accepted the login prompt.

In [ ]:
#from huggingface_hub import login
#login()

#Replace with your own username or organization.
#REPO_ID = 'your-username/KKS-sentiment-workshop'
#raw_ds.push_to_hub(REPO_ID)

After pushing, anyone can load it with:

In [ ]:
# from datasets import load_dataset
# ds = load_dataset('your-username/KSS-sentiment-workshop')
# ds